# Week 5 Exercise - Meeting Notes RAG Knowledge Worker

RAG assistant over local meeting notes with streaming responses, model switching (OpenRouter/Ollama), citations, and SQLite chat history.

In [14]:
# optional installs
# !uv pip install -q langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface chromadb gradio openai python-dotenv

In [15]:
import os
import json
import sqlite3
from datetime import datetime, timezone

import gradio as gr
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [16]:
load_dotenv(override=True)

NOTES_DIR = "/Users/davidkamere/projects/llm_engineering/week5/community-contributions/davidkamere/meeting notes"
VECTOR_DB_DIR = "/Users/davidkamere/projects/llm_engineering/week5/community-contributions/davidkamere/chroma_meeting_notes"
VECTOR_DB_FALLBACK_DIR = "/tmp/chroma_meeting_notes"
CHAT_DB_PATH = "/Users/davidkamere/projects/llm_engineering/week5/community-contributions/davidkamere/week5_chat_history.db"

OPENROUTER_MODEL = "openai/gpt-4o-mini"
OLLAMA_MODEL = "llama3.2"

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [17]:
def init_chat_db():
    with sqlite3.connect(CHAT_DB_PATH) as conn:
        cur = conn.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS chat_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                created_at TEXT NOT NULL,
                model_choice TEXT NOT NULL,
                question TEXT NOT NULL,
                answer TEXT NOT NULL,
                citations_json TEXT NOT NULL
            )
        """)
        conn.commit()

def save_chat_turn(model_choice, question, answer, citations):
    with sqlite3.connect(CHAT_DB_PATH) as conn:
        cur = conn.cursor()
        cur.execute("""
            INSERT INTO chat_history (created_at, model_choice, question, answer, citations_json)
            VALUES (?, ?, ?, ?, ?)
        """, (
            datetime.now(timezone.utc).isoformat(),
            model_choice,
            question,
            answer,
            json.dumps(citations, ensure_ascii=False)
        ))
        conn.commit()

init_chat_db()

In [18]:
def load_and_chunk_notes():
    loader = DirectoryLoader(
        NOTES_DIR,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    docs = loader.load()

    for d in docs:
        src = d.metadata.get("source", "")
        d.metadata["filename"] = os.path.basename(src)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=120
    )
    chunks = splitter.split_documents(docs)
    return chunks

def _can_write_dir(path):
    try:
        os.makedirs(path, exist_ok=True)
        test_file = os.path.join(path, ".write_test")
        with open(test_file, "w", encoding="utf-8") as f:
            f.write("ok")
        os.remove(test_file)
        return True
    except Exception:
        return False

def build_or_load_vectorstore(force_rebuild=False):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    target_dir = VECTOR_DB_DIR if _can_write_dir(VECTOR_DB_DIR) else VECTOR_DB_FALLBACK_DIR
    if target_dir != VECTOR_DB_DIR:
        print(f"Primary vector DB path is not writable. Using fallback: {target_dir}")

    if force_rebuild and os.path.exists(target_dir):
        import shutil
        try:
            shutil.rmtree(target_dir)
        except Exception as e:
            print(f"Rebuild cleanup warning: {e}")

    if os.path.exists(target_dir):
        try:
            return Chroma(persist_directory=target_dir, embedding_function=embeddings)
        except Exception as e:
            print(f"Existing vectorstore open failed ({e}). Rebuilding in fallback dir.")
            target_dir = VECTOR_DB_FALLBACK_DIR

    chunks = load_and_chunk_notes()
    vectordb = Chroma.from_documents(chunks, embedding=embeddings, persist_directory=target_dir)
    return vectordb

vectorstore = build_or_load_vectorstore(force_rebuild=False)
print("Vectorstore ready.")

Vectorstore ready.


In [19]:
def retrieve_context(question, k=5):
    results = vectorstore.similarity_search_with_score(question, k=int(k))
    context_parts = []
    citations = []

    for i, (doc, score) in enumerate(results, start=1):
        filename = doc.metadata.get("filename", "unknown.md")
        snippet = doc.page_content[:260].replace("\n", " ")
        context_parts.append(f"[C{i}] ({filename})\n{doc.page_content}")
        citations.append({
            "chunk": f"C{i}",
            "filename": filename,
            "score": float(score),
            "snippet": snippet
        })

    context = "\n\n".join(context_parts)
    return context, citations

def stream_answer(history, model_choice, top_k):
    if not history or history[-1].get("role") != "user":
        yield history, "No user message found."
        return

    question = history[-1]["content"]
    answer = ""
    trace = ""
    citations = []

    try:
        context, citations = retrieve_context(question, k=top_k)
        trace_lines = [
            f"Retrieved {len(citations)} chunks",
            *[f"- {c.get('chunk','?')} | {c.get('filename','unknown.md')} | score={float(c.get('score', 0.0)):.4f}" for c in citations]
        ]
        trace = "\n".join(trace_lines)

        system_prompt = (
            "You are a meeting-notes knowledge worker. Answer only from provided context. "
            "If context is insufficient, say you are not sure. "
            "Always end with a 'Citations' section listing chunk IDs and filenames used."
        )

        user_prompt = f"Question:\n{question}\n\nContext:\n{context}"
        client = openrouter_client if model_choice == "OpenRouter" else ollama_client
        model = OPENROUTER_MODEL if model_choice == "OpenRouter" else OLLAMA_MODEL

        stream = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.2,
            stream=True
        )

        emitted = False
        for chunk in stream:
            delta = ""
            try:
                delta = chunk.choices[0].delta.content or ""
            except Exception:
                delta = ""
            answer += delta
            emitted = True
            partial = history[:-1] + [{"role": "user", "content": question}, {"role": "assistant", "content": answer}]
            yield partial, trace

        if not emitted:
            fallback = "I could not generate a streamed answer. Please try again."
            partial = history[:-1] + [{"role": "user", "content": question}, {"role": "assistant", "content": fallback}]
            yield partial, trace
            answer = fallback

        try:
            save_chat_turn(model_choice, question, answer, citations)
        except Exception as db_err:
            trace = trace + "\n" + f"History save warning: {db_err}"
            partial = history[:-1] + [{"role": "user", "content": question}, {"role": "assistant", "content": answer}]
            yield partial, trace

    except Exception as e:
        error_answer = f"Error while generating answer: {str(e)}"
        partial = history[:-1] + [{"role": "user", "content": question}, {"role": "assistant", "content": error_answer}]
        trace_msg = (trace + "\n" if trace else "") + f"Pipeline error: {str(e)}"
        yield partial, trace_msg

def add_user_message(message, history):
    history = history or []
    if not (message or "").strip():
        return "", history
    history = history + [{"role": "user", "content": message}]
    return "", history

In [ ]:
with gr.Blocks(title="Meeting Notes RAG Worker") as app:
    gr.Markdown("# Meeting Notes RAG Worker")
    gr.Markdown("Ask questions across your recent meeting notes with citations and retrieval trace.")

    with gr.Row():
        model_choice = gr.Dropdown(["OpenRouter", "Ollama"], value="OpenRouter", label="Model")
        top_k = gr.Slider(minimum=2, maximum=10, value=5, step=1, label="Top-K retrieved chunks")

    chatbot = gr.Chatbot(type="messages", height=420, label="Chat")
    trace_box = gr.Textbox(label="Retrieval Trace", lines=8)

    with gr.Row():
        message = gr.Textbox(label="Your question", placeholder="What did we decide about customer activation?")
        send = gr.Button("Send", variant="primary")

    send.click(
        fn=add_user_message,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    ).then(
        fn=stream_answer,
        inputs=[chatbot, model_choice, top_k],
        outputs=[chatbot, trace_box]
    )

app.launch()

## Evaluation (Week 5 Day 4 style)

Run a small fixed eval set over your meeting-notes RAG worker and score keyword coverage + citation usage.

In [ ]:
EVAL_SET = [
    {
        "question": "What decisions were made about deep dive sessions?",
        "expected_keywords": ["deep dive", "decision", "brief", "debrief"]
    },
    {
        "question": "What recurring blockers came up in Neural-Forge check-ins?",
        "expected_keywords": ["blocker", "neural-forge", "check in"]
    },
    {
        "question": "Summarize major action items from early March meetings.",
        "expected_keywords": ["action", "march", "owner", "due"]
    },
    {
        "question": "What themes appeared in Hope team vs Neural-Forge meetings?",
        "expected_keywords": ["hope", "neural-forge", "theme", "meeting"]
    },
    {
        "question": "What did we discuss on 2026-03-06?",
        "expected_keywords": ["2026-03-06", "deep-dive-brief", "discussion"]
    },
    {
        "question": "What follow-ups should we track from late February debriefs?",
        "expected_keywords": ["follow", "debrief", "february", "track"]
    },
    {
        "question": "Which files were used most in retrieval for strategy-related questions?",
        "expected_keywords": ["file", "retrieval", "strategy"]
    },
    {
        "question": "What open questions remain unresolved across the notes?",
        "expected_keywords": ["open", "unresolved", "question"]
    }
]

def answer_once(question, model_choice="OpenRouter", top_k=5):
    context, citations = retrieve_context(question, k=top_k)

    system_prompt = (
        "You are a meeting-notes knowledge worker. Answer only from provided context. "
        "If context is insufficient, say you are not sure. "
        "Always include a short Citations section."
    )
    user_prompt = f"Question:\n{question}\n\nContext:\n{context}"

    client = openrouter_client if model_choice == "OpenRouter" else ollama_client
    model = OPENROUTER_MODEL if model_choice == "OpenRouter" else OLLAMA_MODEL

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0
    )

    answer = (response.choices[0].message.content or "").strip()
    return answer, citations

def eval_score(answer, citations, expected_keywords):
    text = answer.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in text)
    keyword_score = hits / max(1, len(expected_keywords))
    citation_score = 1.0 if citations else 0.0
    final = 0.8 * keyword_score + 0.2 * citation_score
    return keyword_score, citation_score, final

def run_eval(eval_set=EVAL_SET, model_choice="OpenRouter", top_k=5):
    rows = []
    for i, item in enumerate(eval_set, start=1):
        q = item["question"]
        expected = item["expected_keywords"]
        try:
            answer, citations = answer_once(q, model_choice=model_choice, top_k=top_k)
            ks, cs, fs = eval_score(answer, citations, expected)
            rows.append({
                "id": i,
                "question": q,
                "keyword_score": round(ks, 3),
                "citation_score": round(cs, 3),
                "final_score": round(fs, 3),
                "used_files": ", ".join(sorted({c['filename'] for c in citations}))
            })
        except Exception as e:
            rows.append({
                "id": i,
                "question": q,
                "keyword_score": 0.0,
                "citation_score": 0.0,
                "final_score": 0.0,
                "used_files": f"ERROR: {str(e)}"
            })

    df = pd.DataFrame(rows)
    summary = {
        "model": model_choice,
        "questions": len(df),
        "avg_keyword_score": round(float(df["keyword_score"].mean()), 3),
        "avg_citation_score": round(float(df["citation_score"].mean()), 3),
        "avg_final_score": round(float(df["final_score"].mean()), 3)
    }
    return df, summary

eval_df, eval_summary = run_eval(model_choice="OpenRouter", top_k=5)
display(eval_df)
print("\nEval Summary:")
print(eval_summary)
